# Generación de Predicciones — Smoking Dataset

En este notebook realizaremos la inferencia sobre nuevos datos sin etiquetar. 
Los pasos a seguir son:
1. Carga del modelo optimizado y la lista de variables (features).
2. Carga de los nuevos datos sin etiquetar.
3. Aplicación de las transformaciones necesarias para igualar el formato de entrenamiento.
4. Generación de predicciones.
5. Exportación del archivo resultante con la nueva columna `"smoking_prediction"`.

In [10]:
import os
import pandas as pd
import joblib
import warnings
import numpy as np
warnings.filterwarnings("ignore")

## 2. Carga del Modelo y Features

Cargamos el modelo Random Forest que guardamos en la etapa de optimización y la lista exacta de variables con las que fue entrenado. Esto es crucial para mantener la consistencia en el pipeline.

In [11]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODELS_DIR = os.path.join(BASE_DIR, "models")

modelo_path = os.path.join(MODELS_DIR, "modelo_final.joblib")
features_path = os.path.join(MODELS_DIR, "feature_names.joblib")
scaler_path = os.path.join(MODELS_DIR, "scaler_age.joblib")
# Cargar el modelo entrenado y los nombres de las columnas
modelo_final = joblib.load(modelo_path)
feature_names = joblib.load(features_path)
scaler_full = joblib.load(scaler_path)


print("Artefactos cargados exitosamente:")
print(f"- Modelo: {type(modelo_final).__name__}")
print(f"- Features requeridas: {len(feature_names)}")
print(f"- Scaler: {type(scaler_full).__name__}")

Artefactos cargados exitosamente:
- Modelo: RandomForestClassifier
- Features requeridas: 16
- Scaler: StandardScaler


## 3. Carga de los Datos Sin Etiquetar

Cargamos el dataset que contiene los registros a los cuales queremos predecirles si son fumadores o no. 


In [ ]:
NUEVOS_DATOS_PATH = os.path.join(BASE_DIR, "data", "external", "smoking_prediction_entrega.csv")

df_nuevos = pd.read_csv(NUEVOS_DATOS_PATH)

print(f"Datos cargados. Cantidad de registros a predecir: {df_nuevos.shape[0]}")
df_nuevos.head(3)

Datos cargados. Cantidad de registros a predecir: 5692


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358,M,25,160,65,3.42,0.05,0.00,0.04,0.04,...,3.04,0.63,0.04,0.01,0.75,0.71,0.71,Y,0,Y
1,27364,M,30,180,80,3.46,0.04,0.01,0.04,0.04,...,4.21,0.59,0.04,0.04,0.79,1.13,1.33,Y,0,N
2,27368,M,55,165,60,3.42,0.00,0.01,0.04,0.04,...,2.04,0.63,0.04,0.01,1.08,1.29,2.00,Y,1,Y


In [13]:
rename_map = {
    'height(cm)'         : 'height_cm',
    'weight(kg)'         : 'weight_kg',
    'waist(cm)'          : 'waist_cm',
    'eyesight(left)'     : 'eyesight_left',
    'eyesight(right)'    : 'eyesight_right',
    'hearing(left)'      : 'hearing_left',
    'hearing(right)'     : 'hearing_right',
    'fasting blood sugar': 'fasting_blood_sugar',
    'Urine protein'      : 'urine_protein',
    'serum creatinine'   : 'serum_creatinine',
    'dental caries'      : 'dental_caries',
    'Cholesterol'        : 'cholesterol',
}

df_nuevos.rename(columns=rename_map, inplace=True)

# También bajamos todo a minúsculas para uniformidad
df_nuevos.columns = df_nuevos.columns.str.lower()

print('Columnas finales:')
print(df_nuevos.columns.tolist())

Columnas finales:
['id', 'gender', 'age', 'height_cm', 'weight_kg', 'waist_cm', 'eyesight_left', 'eyesight_right', 'hearing_left', 'hearing_right', 'systolic', 'relaxation', 'fasting_blood_sugar', 'cholesterol', 'triglyceride', 'hdl', 'ldl', 'hemoglobin', 'urine_protein', 'serum_creatinine', 'ast', 'alt', 'gtp', 'oral', 'dental_caries', 'tartar']


## 4. Aplicación de Transformaciones (Pipeline)



In [ ]:
def aplicar_pipeline(df, features_esperadas, scaler):
    df_clean = df.copy()
    
    # Encoding — sin chequear dtype, siempre mapear
    if 'gender' in df_clean.columns:
        df_clean['gender'] = df_clean['gender'].map({'M': 1, 'F': 0, 1: 1, 0: 0})
        
    if 'tartar' in df_clean.columns:
        df_clean['tartar'] = df_clean['tartar'].map({'Y': 1, 'N': 0, 1: 1, 0: 0})

    if 'dental_caries' in df_clean.columns and df_clean['dental_caries'].dtype == 'O':
        df_clean['dental_caries'] = df_clean['dental_caries'].map({'Y': 1, 'N': 0})

    # Asegurar numérico
    df_clean['gender'] = pd.to_numeric(df_clean['gender'], errors='coerce').fillna(0).astype(int)
    df_clean['tartar'] = pd.to_numeric(df_clean['tartar'], errors='coerce').fillna(0).astype(int)
        
    # Feature Engineering
    if all(col in df_clean.columns for col in ['gtp', 'ast', 'alt']):
        df_clean['liver_score'] = df_clean['gtp'] + df_clean['ast'] + df_clean['alt']
        
    if all(col in df_clean.columns for col in ['triglyceride', 'hdl']):
        df_clean['lipid_ratio'] = df_clean['triglyceride'] / df_clean['hdl'].replace(0, np.nan)
        df_clean['lipid_ratio'] = df_clean['lipid_ratio'].fillna(0)
        
    if all(col in df_clean.columns for col in ['gender', 'hemoglobin']):
        df_clean['gender_x_hemoglobin'] = df_clean['gender'] * df_clean['hemoglobin']
        
    # Scaling — solo numéricas continuas
    cols_binarias = ['gender', 'dental_caries', 'tartar', 'smoking']
    cols_to_scale = [
        col for col in features_esperadas 
        if col not in cols_binarias and col in df_clean.columns
    ]
    df_clean[cols_to_scale] = scaler.transform(df_clean[cols_to_scale])
            
    # Verificación
    columnas_faltantes = [col for col in features_esperadas if col not in df_clean.columns]
    if columnas_faltantes:
        raise ValueError(f"Faltan columnas: {columnas_faltantes}")
        
    return df_clean[features_esperadas]

## 5. Generación de las Predicciones

Generamos las predicciones y las probabilidades, y las anexamos al dataframe original.

In [19]:
X_inferencia = aplicar_pipeline(df_nuevos, feature_names, scaler_full)

# Luego esto funciona directo:
predicciones   = modelo_final.predict(X_inferencia)
probabilidades = modelo_final.predict_proba(X_inferencia)[:, 1]

df_final = df_nuevos.copy()
df_final["smoking_prediction"] = predicciones
df_final["smoking_probability"] = probabilidades.round(4)

print("Predicciones generadas con éxito.")
display(df_final[["age", "gender", "smoking_prediction", "smoking_probability"]].head())

Predicciones generadas con éxito.


,age,gender,smoking_prediction,smoking_probability
0,25,M,1,0.8700
1,30,M,1,0.6233
2,55,M,1,0.9600
3,20,M,0,0.2067
4,25,M,0,0.2600


## 6. Exportación del Archivo Resultante

Guardamos el dataframe final con las predicciones en la carpeta correspondiente.

In [ ]:
# Crear directorio de predicciones si no existe
PREDICTIONS_DIR = os.path.join(BASE_DIR, "data", "predictions")
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

export_path = os.path.join(PREDICTIONS_DIR, "predicciones_smoking_.csv")

df_final.to_csv(export_path, index=False)

